# 🔬 Med-StyleGAN Plastic Surgery Simulator
### Face Encoding → Surgical Sliders → Post-Op Preview → Anthropometric Report

---

> ⚠️ **Platform:** This notebook is designed for **Kaggle** (GPU required).
> The dataset used in development is **private** and not included here.
> See the [README](README.md) for instructions on substituting your own data.

---

**Before running:**
1. `Settings → Accelerator → GPU T4 x2` (or P100)
2. `Settings → Internet → ON` *(needed for weight downloads)*
3. Add your own dataset: `+ Add Data` → upload your pre/post-op image folder
4. Update `DATASET_ROOT` in **Block 0** to match your dataset path

**Run order:** Top to bottom. Block 1 = setup (run once, then Restart & Clear Output). Blocks 2–8 = live pipeline.

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════════
# BLOCK 0 ── USER CONFIG  (edit this cell only)
# ═══════════════════════════════════════════════════════════════════════════════
import glob, os

# ── Step 1: See exactly what Kaggle mounted ──────────────────────────────────────
DATASET_ROOT = "/kaggle/input/YOUR-DATASET-SLUG-HERE"

print("── Directory contents (first 30 files) ──")
all_files = glob.glob(os.path.join(DATASET_ROOT, "**", "*"), recursive=True)
for f in sorted(all_files)[:30]:
    print(" ", f)
print(f"\nTotal files found: {len(all_files)}")

# ── Step 2: Auto-detect image folder + naming pattern ───────────────────────────
def find_dataset(root):
    for pat in ["*_a.jpg", "*_a.JPG", "*_a.png",
                "*_before.*", "*_pre.*",
                "*.jpg", "*.JPG", "*.png"]:
        hits = glob.glob(os.path.join(root, "**", pat), recursive=True)
        if hits:
            folder = os.path.dirname(hits[0])
            print(f"\n[✓] Pattern '{pat}' matched in: {folder}")
            print(f"    Sample: {[os.path.basename(h) for h in hits[:5]]}")
            return folder, pat
    return root, "*.jpg"

DATA_DIR, DETECTED_PATTERN = find_dataset(DATASET_ROOT)

# ── Step 3: Set suffixes to match YOUR filenames ─────────────────────────────────
# If your files are named  01_a.jpg / 01_b.jpg  → keep defaults below
# If your files are named  01_before.jpg / 01_after.jpg  → change accordingly
PRE_SUFFIX  = "_a"    # change to "_before", "_pre", etc. if needed
POST_SUFFIX = "_b"    # change to "_after",  "_post", etc. if needed
IMG_EXT     = ".jpg"  # change to ".png" if needed

pre_paths  = sorted(glob.glob(os.path.join(DATA_DIR, f"*{PRE_SUFFIX}{IMG_EXT}")))
post_paths = sorted(glob.glob(os.path.join(DATA_DIR, f"*{POST_SUFFIX}{IMG_EXT}")))

# If still 0, dump all image names so you can identify the correct suffix
if len(pre_paths) == 0:
    all_imgs = sorted(
        glob.glob(os.path.join(DATA_DIR, "**", "*.jpg"), recursive=True) +
        glob.glob(os.path.join(DATA_DIR, "**", "*.png"), recursive=True)
    )
    print(f"\n[!] PRE_SUFFIX='{PRE_SUFFIX}' matched 0 files.")
    print(f"    All images ({len(all_imgs)} total) — set PRE_SUFFIX/POST_SUFFIX to match:")
    for f in all_imgs[:20]:
        print("   ", os.path.basename(f))
    if len(all_imgs) > 20:
        print(f"    ... and {len(all_imgs)-20} more")
else:
    print(f"\n[✓] Pre-op  ({PRE_SUFFIX}): {len(pre_paths)}")
    print(f"[✓] Post-op ({POST_SUFFIX}): {len(post_paths)}")
    print(f"[✓] Pairs: {min(len(pre_paths), len(post_paths))}")

# ── Paths used by all other blocks ───────────────────────────────────────────────
WORK_DIR   = "/kaggle/working"
MODELS_DIR = os.path.join(WORK_DIR, "models")
OUTPUT_DIR = os.path.join(WORK_DIR, "outputs")
REPOS_DIR  = os.path.join(WORK_DIR, "repos")
for d in [MODELS_DIR, OUTPUT_DIR, REPOS_DIR]:
    os.makedirs(d, exist_ok=True)

SLIDER_MIN, SLIDER_MAX = -3.0, 3.0
MAX_PAIRS = 200
DEMO_ID   = "42"

## Block 1 — Environment Setup *(Run Once)*

In [ ]:
# ── Block 1-A: Install Missing Packages ─────────────────────────────────────────
import subprocess, sys, tempfile, os

# ── Lock numpy so nothing can touch it ──────────────────────────────────────────
# scipy / jax / cupy / opencv are compiled against Kaggle's exact numpy build.
# We write a pip constraints file pinning numpy to its current version, then
# install every package with that constraint active. If any package strictly
# requires a different numpy, pip will error instead of silently downgrading.
_np_ver = subprocess.run(
    [sys.executable, "-c", "import numpy; print(numpy.__version__)"],
    capture_output=True, text=True
).stdout.strip()
print(f"[i] Locking numpy at {_np_ver} for all installs")

_cf = tempfile.NamedTemporaryFile(mode="w", suffix=".txt", delete=False)
_cf.write(f"numpy=={_np_ver}\n")
_cf.close()

def pip(*pkgs):
    subprocess.check_call(
        [sys.executable, "-m", "pip", "install", "-q", "-c", _cf.name] + list(pkgs)
    )

pip("ninja")                              # StyleGAN2 custom CUDA kernel compiler
pip("mediapipe>=0.10.14,<=0.10.18")       # FaceMesh; this range supports numpy 2.x
pip("dlib")                               # face alignment (~2 min compile on Kaggle)
pip("ultralytics")                        # YOLOv8 face detector
pip("ipywidgets", "--upgrade")

os.unlink(_cf.name)

# ── Verify numpy was not changed ─────────────────────────────────────────────────
_np_after = subprocess.run(
    [sys.executable, "-c", "import numpy; print(numpy.__version__)"],
    capture_output=True, text=True
).stdout.strip()
status = "OK — unchanged" if _np_after == _np_ver else f"CHANGED to {_np_after} — restart may still be needed"
print(f"[✓] numpy {_np_after}: {status}")
print("\n[✓] All packages installed.")
print("=" * 60)
print("  NEXT: Run → Restart & Clear Output")
print("  Then re-run ALL cells from Block 0 downward.")
print("=" * 60)

In [ ]:
# ── Block 1-B: Clone Repos ──────────────────────────────────────────────────────
import os

E4E_DIR = os.path.join(REPOS_DIR, "encoder4editing")
SG2_DIR = os.path.join(REPOS_DIR, "stylegan2-ada-pytorch")

def git_clone(url, dest):
    if not os.path.exists(dest):
        ret = os.system(f"git clone --depth 1 -q {url} {dest}")
        print(f"[✓] Cloned {os.path.basename(dest)}" if ret == 0
              else f"[!] Clone failed: {url}")
    else:
        print(f"[✓] Exists: {os.path.basename(dest)}")

git_clone("https://github.com/omertov/encoder4editing.git",       E4E_DIR)
git_clone("https://github.com/NVlabs/stylegan2-ada-pytorch.git",  SG2_DIR)

import sys
for p in [E4E_DIR, SG2_DIR]:
    if p not in sys.path:
        sys.path.insert(0, p)

print("[✓] Repos in sys.path")

In [ ]:
# ── Block 1-C: Download Weights ─────────────────────────────────────────────────
# Weights saved to /kaggle/working/models/ (persists for session)

import gdown, urllib.request, bz2, requests, os, numpy as np

E4E_WEIGHTS  = os.path.join(MODELS_DIR, "e4e_ffhq_encode.pt")
SG2_WEIGHTS  = os.path.join(MODELS_DIR, "stylegan2-ffhq.pkl")
GANSPACE_NPY = os.path.join(MODELS_DIR, "ganspace_pca_ffhq.npy")
SHAPE_PRED   = os.path.join(MODELS_DIR, "shape_predictor_68_face_landmarks.dat")

# ── e4e encoder (Google Drive, ~1 GB) ────────────────────────────────────────────
if not os.path.exists(E4E_WEIGHTS):
    print("Downloading e4e weights (~1 GB)...")
    gdown.download(id="1cUv_reLE6k3604or78EranS7XzuVMWeO",
                   output=E4E_WEIGHTS, quiet=False)
else:
    print("[✓] e4e weights")

# ── StyleGAN2-FFHQ generator pkl (~300 MB) ───────────────────────────────────────
if not os.path.exists(SG2_WEIGHTS):
    print("Downloading StyleGAN2-FFHQ (~300 MB)...")
    url = "https://nvlabs-fi-cdn.nvidia.com/stylegan2-ada-pytorch/pretrained/ffhq.pkl"
    r = requests.get(url, stream=True)
    with open(SG2_WEIGHTS, "wb") as f:
        for chunk in r.iter_content(8192):
            f.write(chunk)
else:
    print("[✓] StyleGAN2 pkl")

# ── dlib 68-point predictor (~100 MB) ────────────────────────────────────────────
SHAPE_BZ2 = SHAPE_PRED + ".bz2"
if not os.path.exists(SHAPE_PRED):
    print("Downloading dlib shape predictor...")
    urllib.request.urlretrieve(
        "http://dlib.net/files/shape_predictor_68_face_landmarks.dat.bz2", SHAPE_BZ2
    )
    with bz2.open(SHAPE_BZ2) as fin, open(SHAPE_PRED, "wb") as fout:
        fout.write(fin.read())
    os.remove(SHAPE_BZ2)
else:
    print("[✓] dlib predictor")

# ── GANSpace PCA directions ───────────────────────────────────────────────────────
if not os.path.exists(GANSPACE_NPY):
    print("Downloading GANSpace directions...")
    try:
        gdown.download(id="1N2tgmnatZJ4b4gM9GiCDGDPN7g3GdSLM",
                       output=GANSPACE_NPY, quiet=False)
    except Exception:
        print("  [WARN] GANSpace download failed — using synthetic orthogonal basis")
        # Use Python's built-in random to avoid numpy.random binary-incompatibility crash
        import random as _rnd
        _rnd.seed(42)
        A = np.array([[_rnd.gauss(0.0, 1.0) for _ in range(512)]
                      for _ in range(512)], dtype=np.float32)
        Q, _ = np.linalg.qr(A)
        np.save(GANSPACE_NPY, Q.astype(np.float32))
else:
    print("[✓] GANSpace directions")

print("\n[✓] All weights ready")

## Block 2 — Core Imports & GPU Check

In [ ]:
# ── Block 2: Imports + Device ────────────────────────────────────────────────────
import torch, numpy as np, cv2, dlib
import matplotlib.pyplot as plt, matplotlib.patches as mpatches
import matplotlib.gridspec as gridspec
import ipywidgets as widgets
import pandas as pd
from IPython.display import display, clear_output
from PIL import Image
from scipy.spatial.distance import euclidean
from tqdm.notebook import tqdm
import os, glob, random, warnings, scipy
warnings.filterwarnings("ignore")

# mediapipe top-level import can fail in some 0.10.x builds due to an
# audio/__init__.py NameError — the solutions we need are imported directly
# in Block 5-A, so a failure here is non-fatal.
try:
    import mediapipe as mp
    print("[✓] mediapipe top-level import OK")
except Exception as _e:
    mp = None
    print(f"[WARN] mediapipe top-level import failed: {_e}")
    print("       FaceMesh will be imported directly in Block 5-A — OK to continue.")

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

if torch.cuda.is_available():
    gpu = torch.cuda.get_device_name(0)
    vram = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f"[✓] GPU: {gpu}  ({vram:.1f} GB VRAM)")
else:
    print("[!] No GPU — GAN synthesis will be slow. Enable GPU in Settings.")

print(f"[✓] PyTorch {torch.__version__} | device = {DEVICE}")
print(f"[✓] Dataset: {len(glob.glob(os.path.join(DATA_DIR, '*_a.jpg')))} pre-op images found")

## Block 3 — Identity Encoding Pipeline

In [ ]:
# ── Block 3-A: YOLO / dlib Face Detector ────────────────────────────────────────

class FaceDetector:
    """YOLOv8-face → dlib HOG fallback."""
    def __init__(self):
        self.backend = None
        try:
            from ultralytics import YOLO
            self.yolo = YOLO("yolov8n-face.pt")
            self.backend = "yolo"
            print("[✓] Detector: YOLOv8-face")
        except Exception as e:
            self.dlib_det = dlib.get_frontal_face_detector()
            self.backend = "dlib"
            print(f"[✓] Detector: dlib HOG ({e})")

    def detect(self, bgr):
        if self.backend == "yolo":
            res = self.yolo(bgr, verbose=False)[0]
            return [tuple(map(int, b[:4])) for b in res.boxes.xyxy.cpu().numpy()]
        gray = cv2.cvtColor(bgr, cv2.COLOR_BGR2GRAY)
        return [(d.left(), d.top(), d.right(), d.bottom())
                for d in self.dlib_det(gray, 1)]

    def crop_largest(self, bgr, pad=0.25):
        boxes = self.detect(bgr)
        if not boxes:
            return bgr
        x1, y1, x2, y2 = max(boxes, key=lambda b: (b[2]-b[0])*(b[3]-b[1]))
        h, w = bgr.shape[:2]
        pw, ph = int((x2-x1)*pad), int((y2-y1)*pad)
        return bgr[max(0,y1-ph):min(h,y2+ph), max(0,x1-pw):min(w,x2+pw)]


# ── Block 3-B: Face Aligner (FFHQ protocol) ─────────────────────────────────────

class FaceAligner:
    """Aligns face to FFHQ canonical position (eyes at 35%/65% x, 35% y) → 1024×1024."""
    LEFT_EYE  = np.array([0.35, 0.35])
    RIGHT_EYE = np.array([0.65, 0.35])
    SIZE = 1024

    def __init__(self, predictor_path):
        self._ok = False
        if os.path.exists(predictor_path):
            self.det  = dlib.get_frontal_face_detector()
            self.pred = dlib.shape_predictor(predictor_path)
            self._ok  = True
            print("[✓] FaceAligner: dlib 68-pt")
        else:
            print("[WARN] FaceAligner: resize-only mode")

    def align(self, rgb):
        if not self._ok:
            return cv2.resize(rgb, (self.SIZE, self.SIZE))
        gray = cv2.cvtColor(rgb, cv2.COLOR_RGB2GRAY)
        dets = self.det(gray, 1)
        if not dets:
            return cv2.resize(rgb, (self.SIZE, self.SIZE))
        det   = max(dets, key=lambda d: d.width()*d.height())
        shape = self.pred(gray, det)
        lc = np.mean([(shape.part(i).x, shape.part(i).y) for i in range(36, 42)], 0)
        rc = np.mean([(shape.part(i).x, shape.part(i).y) for i in range(42, 48)], 0)
        tl, tr  = self.LEFT_EYE * self.SIZE, self.RIGHT_EYE * self.SIZE
        angle   = np.degrees(np.arctan2(tr[1]-tl[1], tr[0]-tl[0])) - \
                  np.degrees(np.arctan2(rc[1]-lc[1], rc[0]-lc[0]))
        scale   = np.linalg.norm(tr-tl) / (np.linalg.norm(rc-lc) + 1e-8)
        ec_src  = ((lc+rc)/2).astype(np.float32)
        ec_tgt  = ((tl+tr)/2).astype(np.float32)
        M = cv2.getRotationMatrix2D(tuple(ec_src),
                                     np.degrees(np.arctan2(rc[1]-lc[1], rc[0]-lc[0])),
                                     scale)
        M[0,2] += ec_tgt[0] - ec_src[0]
        M[1,2] += ec_tgt[1] - ec_src[1]
        return cv2.warpAffine(rgb, M, (self.SIZE, self.SIZE),
                              flags=cv2.INTER_CUBIC, borderMode=cv2.BORDER_REFLECT)


face_detector = FaceDetector()
face_aligner  = FaceAligner(SHAPE_PRED)

In [ ]:
# ── Block 3-C: e4e Encoder (Image → 18×512 W+ latent) ───────────────────────────

class E4EEncoder:
    def __init__(self, weights_path, device):
        self.device = device
        self.model  = None
        if not os.path.exists(weights_path):
            print("[WARN] e4e: stub mode (weights missing)")
            return
        try:
            from argparse import Namespace
            ckpt = torch.load(weights_path, map_location=device)
            opts = Namespace(**ckpt["opts"])
            opts.device = device
            opts.checkpoint_path = weights_path
            from models.psp import pSp
            self.model = pSp(opts).to(device).eval()
            print(f"[✓] e4e encoder on {device}")
        except Exception as e:
            print(f"[WARN] e4e load failed: {e}")

    def encode(self, img_rgb_1024):
        """Returns (n_ws, w_dim) W+ latent tensor, typically (18, 512)."""
        if self.model is None:
            return torch.zeros(18, 512)
        x = torch.from_numpy(
            ((img_rgb_1024.astype(np.float32)/255.0 - 0.5)/0.5)
            .transpose(2,0,1)
        ).unsqueeze(0).to(self.device)
        with torch.no_grad():
            # pSp.forward() returns (images, latent_codes) — latent is 2nd value
            _, codes = self.model(x, randomize_noise=False, return_latents=True)
        codes = codes.squeeze(0).cpu()   # (1, n_ws, w_dim) → (n_ws, w_dim)
        if codes.dim() != 2:             # guard against unexpected extra dims
            codes = codes.reshape(-1, codes.shape[-1])
        return codes


# ── Block 3-D: StyleGAN2 Generator (W+ → 1024×1024 image) ───────────────────────

class StyleGAN2Generator:
    def __init__(self, pkl_path, device):
        self.device    = device
        self.G         = None
        self._stub_img = None
        if not os.path.exists(pkl_path):
            print("[WARN] StyleGAN2: stub mode (pkl missing)")
            return
        try:
            import pickle
            with open(pkl_path, "rb") as f:
                self.G = pickle.load(f)["G_ema"].to(device).eval()
            print(f"[✓] StyleGAN2 generator on {device}")
        except Exception as e:
            print(f"[WARN] SG2 load failed: {e}")

    def synthesize(self, w_18x512):
        """Returns 1024×1024 uint8 RGB numpy array."""
        if self.G is None:
            return cv2.GaussianBlur(self._stub_img, (0,0), 1.5) \
                   if self._stub_img is not None \
                   else np.zeros((1024,1024,3), dtype=np.uint8)
        w = w_18x512
        if w.dim() != 2:
            w = w.reshape(-1, w.shape[-1])
        # Conform to the number of style vectors the loaded generator expects
        expected_ws = self.G.mapping.num_ws
        if w.shape[0] != expected_ws:
            print(f"[WARN] w has {w.shape[0]} style vecs, generator expects {expected_ws} — adjusting")
            if w.shape[0] > expected_ws:
                w = w[:expected_ws]
            else:
                w = w.repeat(expected_ws, 1)[:expected_ws]
        ws = w.unsqueeze(0).to(self.device)
        with torch.no_grad():
            t = self.G.synthesis(ws, noise_mode="const")
        img = t.squeeze(0).permute(1,2,0).cpu().numpy()
        return ((img + 1) * 127.5).clip(0, 255).astype(np.uint8)


encoder   = E4EEncoder(E4E_WEIGHTS,  DEVICE)
generator = StyleGAN2Generator(SG2_WEIGHTS, DEVICE)

## Block 4 — Surgical Controls: GANSpace Directions + Vector Math

In [ ]:
# ── Block 4: GANSpace Directions + apply_surgical_edits() ───────────────────────

ganspace_dirs = np.load(GANSPACE_NPY)
# Normalise to 2-D (N_components, latent_dim) regardless of how the file was saved
if ganspace_dirs.ndim == 3:               # e.g. (N, n_layers, 512) from some GANSpace saves
    ganspace_dirs = ganspace_dirs[:, 0, :]
print(f"[✓] GANSpace directions: {ganspace_dirs.shape}")

# Slider name → (PC index, w_layer_start, w_layer_end, scale)
SURGICAL = {
    "Nose Bridge (Hump)"    : (10,  4,  8, 1.5),
    "Nose Tip Rotation"     : (11,  5,  9, 1.2),
    "Alar Width (Flare)"    : (12,  4,  8, 1.0),
    "Jawline Width (V-line)": (14,  8, 14, 1.5),
    "Chin Projection"       : (15,  9, 14, 1.2),
    "Cheek Volume"          : (16,  6, 12, 1.0),
    "Eye Opening (Bleph.)"  : ( 7,  3,  7, 1.0),
    "Lip Fullness"          : (17,  9, 13, 1.0),
    "Face Length"           : ( 5,  2,  6, 1.5),
    "Forehead Slope"        : ( 3,  2,  5, 1.0),
}

def apply_surgical_edits(w_base, slider_values):
    """
    w_base       : (18, 512) tensor — or any shape; normalised internally
    slider_values: {name: float}
    Returns      : (18, latent_dim) edited latent
    """
    w = w_base.clone()

    # ── Normalise to 2-D (n_layers, latent_dim) ──────────────────────────────
    # The e4e model may return (1, 18, 512), (18, 1, 512), or other shapes.
    if w.dim() != 2:
        w = w.reshape(-1, w.shape[-1])   # collapse leading / extra dims
    n_layers, latent_dim = w.shape

    for name, val in slider_values.items():
        if name not in SURGICAL or abs(val) < 1e-6:
            continue
        idx, s, e, scale = SURGICAL[name]
        # ── Build direction vector matching latent_dim ────────────────────────
        d_np = ganspace_dirs[idx].flatten()          # always 1-D after flatten
        if len(d_np) >= latent_dim:
            d = torch.from_numpy(d_np[:latent_dim]).float()
        else:
            # pad with zeros if direction is shorter (shouldn't happen normally)
            d = torch.zeros(latent_dim)
            d[:len(d_np)] = torch.from_numpy(d_np).float()
        for layer in range(s, min(e, n_layers)):
            w[layer] += val * scale * d
    return w

def full_pipeline(img_path_or_rgb, slider_values):
    """Returns (pre_op_rgb, post_op_rgb) both 1024×1024."""
    if isinstance(img_path_or_rgb, str):
        bgr = cv2.imread(img_path_or_rgb)
        rgb = cv2.cvtColor(bgr, cv2.COLOR_BGR2RGB)
    else:
        rgb = img_path_or_rgb
    crop    = face_detector.crop_largest(cv2.cvtColor(rgb, cv2.COLOR_RGB2BGR))
    pre_op  = face_aligner.align(cv2.cvtColor(crop, cv2.COLOR_BGR2RGB))
    w_base  = encoder.encode(pre_op)
    w_new   = apply_surgical_edits(w_base, slider_values)
    generator._stub_img = pre_op
    post_op = generator.synthesize(w_new)
    return pre_op, post_op

# Sanity check
w_test = apply_surgical_edits(torch.zeros(18, 512), {"Nose Bridge (Hump)": 1.5})
print(f"[✓] Vector math: ||delta|| = {(w_test - torch.zeros(18, 512)).norm():.4f}")

## Block 5 — Anthropometric Measurement (MediaPipe 468 Landmarks)

In [ ]:
# ── Block 5-A: AnthropometricAnalyzer ───────────────────────────────────────────

# Resolve mediapipe solutions across all 0.10.x wheel layouts
try:
    import mediapipe.solutions.face_mesh as mp_face_mesh
    import mediapipe.solutions.drawing_utils as mp_draw
except ModuleNotFoundError:
    try:
        import mediapipe.python.solutions.face_mesh as mp_face_mesh
        import mediapipe.python.solutions.drawing_utils as mp_draw
    except ModuleNotFoundError:
        # Last resort: trigger lazy subpackage registration then retry
        import mediapipe as _mp
        import importlib
        mp_face_mesh = importlib.import_module("mediapipe.solutions.face_mesh")
        mp_draw      = importlib.import_module("mediapipe.solutions.drawing_utils")

print(f"[✓] mediapipe face_mesh: {mp_face_mesh.__name__}")

from scipy.spatial.distance import euclidean
import numpy as np, cv2, os, pandas as pd

class AnthropometricAnalyzer:
    LM = {
        "nose_tip":4, "nose_bridge_top":6, "nose_bridge_mid":197,
        "philtrum":164, "alar_left":129, "alar_right":358,
        "chin_tip":152, "jaw_left":234,  "jaw_right":454,
        "jaw_gonion_L":172, "jaw_gonion_R":397,
        "leye_top":159, "leye_bot":145, "reye_top":386, "reye_bot":374,
        "leye_out":33,  "leye_in":133,  "reye_out":263, "reye_in":362,
        "lip_top":13,   "lip_bot":14,   "lip_L":61,     "lip_R":291,
        "forehead":10,  "lcheek":116,   "rcheek":345,
    }

    def __init__(self):
        self.mesh = mp_face_mesh.FaceMesh(
            static_image_mode=True, max_num_faces=1,
            refine_landmarks=True, min_detection_confidence=0.4
        )

    def get_landmarks(self, rgb):
        h, w = rgb.shape[:2]
        res  = self.mesh.process(rgb)
        if not res.multi_face_landmarks:
            return None
        lms = res.multi_face_landmarks[0].landmark
        return {k: np.array([lms[i].x*w, lms[i].y*h, lms[i].z*w])
                for k, i in self.LM.items()}

    def compute_metrics(self, lm):
        if lm is None:
            return {}
        d2 = lambda a,b: euclidean(lm[a][:2], lm[b][:2])
        d3 = lambda a,b: euclidean(lm[a],      lm[b])
        IOD  = d2("leye_out", "reye_out") + 1e-8

        v1   = lm["nose_tip"][:2] - lm["philtrum"][:2]
        cos_a = np.dot(v1, [0,-1]) / (np.linalg.norm(v1) + 1e-8)
        nla   = float(np.degrees(np.arccos(np.clip(cos_a, -1, 1))))

        return {
            "Nose Length"       : d2("nose_bridge_top", "nose_tip")     / IOD,
            "Nose Tip Protrusion": d3("nose_tip",       "philtrum"),
            "Alar Base Width"   : d2("alar_left",       "alar_right")   / IOD,
            "Nasolabial Angle°" : nla,
            "Jaw Width"         : d2("jaw_gonion_L",    "jaw_gonion_R") / IOD,
            "Face Width"        : d2("jaw_left",        "jaw_right")    / IOD,
            "Chin Height"       : d2("philtrum",        "chin_tip")     / IOD,
            "Chin Projection Δ" : float(lm["chin_tip"][2] - lm["nose_tip"][2]),
            "L Eye Aperture"    : d2("leye_top",        "leye_bot")     / IOD,
            "R Eye Aperture"    : d2("reye_top",        "reye_bot")     / IOD,
            "Intercanthal Dist": d2("leye_in",         "reye_in")      / IOD,
            "Lip Height"        : d2("lip_top",         "lip_bot")      / IOD,
            "Lip Width"         : d2("lip_L",           "lip_R")        / IOD,
            "Face Length"       : d2("forehead",        "chin_tip")     / IOD,
            "Cheek Width"       : d2("lcheek",          "rcheek")       / IOD,
            "Golden Ratio (W/L)": d2("jaw_left","jaw_right") /
                                  (d2("forehead","chin_tip") + 1e-8),
        }

    def analyze_pair(self, pre, post):
        lm_pre  = self.get_landmarks(pre)
        lm_post = self.get_landmarks(post)
        pm, qm  = self.compute_metrics(lm_pre), self.compute_metrics(lm_post)
        if not pm or not qm:
            return pm, qm, None
        rows = []
        for k in pm:
            pv, qv = pm[k], qm.get(k, np.nan)
            dv = qv - pv
            rows.append({"Metric":k, "Pre-Op":round(pv,4), "Post-Op":round(qv,4),
                         "Δ (abs)":round(dv,4),
                         "Δ (%)":round(dv/abs(pv)*100 if abs(pv)>1e-6 else 0, 2)})
        return pm, qm, pd.DataFrame(rows).set_index("Metric")

    def draw_mesh(self, rgb):
        out = rgb.copy()
        res = self.mesh.process(rgb)
        if res.multi_face_landmarks:
            for fl in res.multi_face_landmarks:
                mp_draw.draw_landmarks(
                    out, fl, mp_face_mesh.FACEMESH_TESSELATION, None,
                    mp_draw.DrawingSpec(color=(0,200,100), thickness=1, circle_radius=0))
        return out

analyzer = AnthropometricAnalyzer()
print("[✓] AnthropometricAnalyzer ready — 16 clinical metrics")

In [ ]:
# ── Block 5-B: render_medical_report() ──────────────────────────────────────────

def render_medical_report(pre, post, delta_df=None, title="", save_path=None):
    """3-panel: [Pre+Mesh] | [Post+Mesh] | [Δ bar chart]"""
    pre_m  = analyzer.draw_mesh(pre)
    post_m = analyzer.draw_mesh(post)

    if delta_df is not None and len(delta_df):
        fig = plt.figure(figsize=(22, 9))
        gs  = fig.add_gridspec(1, 3, width_ratios=[2, 2, 3])
        ax1 = fig.add_subplot(gs[0])
        ax2 = fig.add_subplot(gs[1])
        ax3 = fig.add_subplot(gs[2])
    else:
        fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 8))
        ax3 = None

    SZ = 512
    ax1.imshow(cv2.resize(pre_m,  (SZ, SZ))); ax1.axis("off")
    ax1.set_title("Pre-Op + Landmarks",  fontsize=13, fontweight="bold", color="#2c7bb6")
    ax2.imshow(cv2.resize(post_m, (SZ, SZ))); ax2.axis("off")
    ax2.set_title("Post-Op + Landmarks", fontsize=13, fontweight="bold", color="#d7191c")

    if ax3 is not None:
        deltas = delta_df["Δ (abs)"].astype(float)
        colors = ["#d7191c" if v > 0 else "#2c7bb6" for v in deltas]
        ax3.barh(range(len(deltas)), deltas.values, color=colors, alpha=0.85, edgecolor="white")
        ax3.set_yticks(range(len(deltas)))
        ax3.set_yticklabels(deltas.index, fontsize=9)
        ax3.axvline(0, color="black", lw=0.8, ls="--")
        ax3.set_xlabel("Δ (normalized)", fontsize=10)
        ax3.set_title("Anthropometric Changes", fontsize=12, fontweight="bold")
        for i, (v, pct) in enumerate(zip(delta_df["Δ (abs)"], delta_df["Δ (%)"] )):
            ax3.text(float(v) + (0.003 if float(v)>=0 else -0.003), i,
                     f"{pct:+.1f}%", va="center",
                     ha="left" if float(v)>=0 else "right", fontsize=7.5)
        ax3.legend(handles=[
            mpatches.Patch(color="#d7191c", label="Increase"),
            mpatches.Patch(color="#2c7bb6", label="Decrease")
        ], fontsize=9, loc="lower right")

    if title:
        fig.suptitle(title, fontsize=14, fontweight="bold", y=1.01)
    plt.tight_layout()
    if save_path:
        plt.savefig(save_path, dpi=120, bbox_inches="tight")
    plt.show()

print("[✓] render_medical_report() ready")

## Block 6 — Live Surgical Dashboard *(ipywidgets)*

In [ ]:
# ── Block 6: Interactive Surgical Simulator ──────────────────────────────────────
# Kaggle note: FileUpload not available — use dataset dropdown instead.

all_pre_paths = sorted(glob.glob(os.path.join(DATA_DIR, "*_a.jpg")))

_state = {"pre_img": None, "w_base": None}

# ─ Selector ─────────────────────────────────────────────────────────────────────
selector = widgets.Dropdown(
    options=[(os.path.basename(p).replace("_a.jpg",""), p) for p in all_pre_paths[:100]],
    description="Select pair:",
    layout=widgets.Layout(width="360px")
)
load_btn = widgets.Button(description="Load & Encode", button_style="success",
                          layout=widgets.Layout(width="140px"))
status   = widgets.Label("Select a pair and click Load & Encode")

# ─ 10 Surgical Sliders ──────────────────────────────────────────────────────────
sliders = {}
for name in SURGICAL:
    sliders[name] = widgets.FloatSlider(
        value=0.0, min=SLIDER_MIN, max=SLIDER_MAX, step=0.1,
        description=name[:24],
        style={"description_width": "195px"},
        layout=widgets.Layout(width="500px"),
        continuous_update=False,
        readout_format=".1f"
    )

reset_btn    = widgets.Button(description="Reset Sliders",    button_style="warning")
generate_btn = widgets.Button(description="Generate Post-Op", button_style="danger",
                               layout=widgets.Layout(width="170px"))
output_area  = widgets.Output()

# ─ Callbacks ────────────────────────────────────────────────────────────────────
def _encode(path):
    bgr  = cv2.imread(path)
    rgb  = cv2.cvtColor(bgr, cv2.COLOR_BGR2RGB)
    crop = face_detector.crop_largest(cv2.cvtColor(rgb, cv2.COLOR_RGB2BGR))
    pre  = face_aligner.align(cv2.cvtColor(crop, cv2.COLOR_BGR2RGB))
    _state["pre_img"] = pre
    _state["w_base"]  = encoder.encode(pre)
    status.value = f"[✓] Loaded & encoded: {os.path.basename(path)}"

def on_load(b):
    status.value = "Encoding..."
    _encode(selector.value)
load_btn.on_click(on_load)

def on_reset(b):
    for s in sliders.values():
        s.value = 0.0
reset_btn.on_click(on_reset)

def on_generate(b):
    with output_area:
        clear_output(wait=True)
        if _state["pre_img"] is None:
            print("[!] Load an image first.")
            return
        sv      = {k: s.value for k, s in sliders.items()}
        w_new   = apply_surgical_edits(_state["w_base"], sv)
        generator._stub_img = _state["pre_img"]
        post    = generator.synthesize(w_new)
        _, _, df = analyzer.analyze_pair(_state["pre_img"], post)
        active  = {k: v for k, v in sv.items() if abs(v) > 0.05}
        title   = "Simulation" + (" | " + ", ".join(f"{k[:14]}={v:+.1f}"
                                                     for k,v in active.items())
                                   if active else "")
        sp = os.path.join(OUTPUT_DIR, "latest_report.png")
        render_medical_report(_state["pre_img"], post, df, title=title, save_path=sp)
        if df is not None:
            display(df.style
                .applymap(lambda v: "color:#d7191c" if float(v)>0 else "color:#2c7bb6",
                          subset=["Δ (%)"])
                .format({"Pre-Op":"{:.4f}","Post-Op":"{:.4f}",
                         "Δ (abs)":"{:+.4f}","Δ (%)":"{:+.2f}%"})
            )
            df.to_csv(os.path.join(OUTPUT_DIR, "metrics_latest.csv"))
        status.value = f"[✓] Report saved to {sp}"
generate_btn.on_click(on_generate)

# ─ Assemble UI ───────────────────────────────────────────────────────────────────
display(widgets.VBox([
    widgets.HTML("<h3 style='margin:4px 0'>🔬 Med-StyleGAN Surgical Simulator (Kaggle)</h3><hr>"),
    widgets.HBox([selector, load_btn]),
    status,
    widgets.HTML("<b>Surgical Controls — drag slider, then click Generate</b>"),
    widgets.VBox(list(sliders.values()),
                 layout=widgets.Layout(border="1px solid #ccc", padding="10px")),
    widgets.HBox([reset_btn, generate_btn]),
    output_area
]))

## Block 7 — Dataset-Level Before/After Analysis (662 Pairs)

In [ ]:
# ── Block 7-A: Batch Metric Extraction ──────────────────────────────────────────
# Processes up to MAX_PAIRS pairs. Each image resized to 512 for speed.

pre_paths  = sorted(glob.glob(os.path.join(DATA_DIR, "*_a.jpg")))
post_paths = sorted(glob.glob(os.path.join(DATA_DIR, "*_b.jpg")))
if MAX_PAIRS:
    pre_paths  = pre_paths[:MAX_PAIRS]
    post_paths = post_paths[:MAX_PAIRS]

print(f"Processing {len(pre_paths)} pairs (resize to 512 for speed)...")

all_pre, all_post, pair_ids, failed = [], [], [], 0

for pp, qp in tqdm(zip(pre_paths, post_paths), total=len(pre_paths)):
    pid = os.path.basename(pp).replace("_a.jpg", "")
    pb  = cv2.imread(pp);  qb = cv2.imread(qp)
    if pb is None or qb is None:
        failed += 1; continue
    pr  = cv2.resize(cv2.cvtColor(pb, cv2.COLOR_BGR2RGB), (512, 512))
    qr  = cv2.resize(cv2.cvtColor(qb, cv2.COLOR_BGR2RGB), (512, 512))
    lp  = analyzer.get_landmarks(pr)
    lq  = analyzer.get_landmarks(qr)
    mp_ = analyzer.compute_metrics(lp)
    mq_ = analyzer.compute_metrics(lq)
    if mp_ and mq_:
        all_pre.append(mp_); all_post.append(mq_); pair_ids.append(pid)
    else:
        failed += 1

df_pre   = pd.DataFrame(all_pre,  index=pair_ids)
df_post  = pd.DataFrame(all_post, index=pair_ids)
df_delta = df_post - df_pre

print(f"[✓] Processed: {len(pair_ids)} | Failed: {failed}")
df_delta.head(3)

In [ ]:
# ── Block 7-B: Population Statistics ────────────────────────────────────────────

from scipy import stats as spstats

summary = pd.DataFrame({
    "Pre-Op Mean"  : df_pre.mean(),
    "Post-Op Mean" : df_post.mean(),
    "Δ Mean"       : df_delta.mean(),
    "Δ Std"        : df_delta.std(),
    "Δ % change"   : (df_delta / df_pre.abs().replace(0, np.nan) * 100).mean(),
    "p-value (t)"  : [spstats.ttest_rel(df_pre[c], df_post[c]).pvalue
                      for c in df_pre.columns]
})
summary["Sig (p<.05)"] = summary["p-value (t)"] < 0.05

print("── Population-Level Surgical Statistics ──")
display(summary.round(4).style
    .applymap(lambda v: "color:#d7191c;font-weight:bold" if v else "",
              subset=["Sig (p<.05)"])
)
summary.to_csv(os.path.join(OUTPUT_DIR, "population_stats.csv"))
print(f"[✓] Saved → {OUTPUT_DIR}/population_stats.csv")

In [ ]:
# ── Block 7-C: Distribution Plots ───────────────────────────────────────────────

KEY = [c for c in [
    "Nose Length", "Alar Base Width", "Nasolabial Angle°",
    "Jaw Width",   "Chin Height",     "Lip Height",
    "L Eye Aperture", "Golden Ratio (W/L)"
] if c in df_delta.columns]

fig, axes = plt.subplots(2, 4, figsize=(20, 9))
for i, metric in enumerate(KEY[:8]):
    ax = axes.flatten()[i]
    ax.hist(df_pre[metric].dropna(),  20, alpha=.55, color="#2c7bb6", label="Pre",  ec="white")
    ax.hist(df_post[metric].dropna(), 20, alpha=.55, color="#d7191c", label="Post", ec="white")
    ax.axvline(df_pre[metric].mean(),  color="#2c7bb6", lw=2, ls="--")
    ax.axvline(df_post[metric].mean(), color="#d7191c", lw=2, ls="--")
    pv   = summary.loc[metric, "p-value (t)"] if metric in summary.index else 1
    ax.set_title(f"{metric[:28]}{' *' if pv<0.05 else ''}", fontsize=9, fontweight="bold")
    ax.legend(fontsize=7); ax.tick_params(labelsize=7)
    # delta inset
    ai = ax.inset_axes([0.62, 0.55, 0.36, 0.38])
    ai.hist(df_delta[metric].dropna(), 15, color="#1a9641", alpha=.75, ec="white")
    ai.axvline(0, color="black", lw=.8); ai.set_title("Δ", fontsize=7)
    ai.tick_params(labelsize=6)

plt.suptitle(f"Anthropometric Distributions — {len(pair_ids)} Surgical Pairs",
             fontsize=13, fontweight="bold", y=1.01)
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, "distributions.png"), dpi=120, bbox_inches="tight")
plt.show()
print("[✓] Saved → outputs/distributions.png")

In [ ]:
# ── Block 7-D: Correlation Heatmap ──────────────────────────────────────────────

corr = df_delta[KEY].corr()
fig, ax = plt.subplots(figsize=(10, 8))
im = ax.imshow(corr.values, cmap="RdBu_r", vmin=-1, vmax=1)
labels = [m[:22] for m in corr.columns]
ax.set_xticks(range(len(labels))); ax.set_xticklabels(labels, rotation=40, ha="right", fontsize=8)
ax.set_yticks(range(len(labels))); ax.set_yticklabels(labels, fontsize=8)
for i in range(len(corr)):
    for j in range(len(corr)):
        v = corr.values[i,j]
        ax.text(j, i, f"{v:.2f}", ha="center", va="center",
                fontsize=7, color="white" if abs(v)>.5 else "black")
plt.colorbar(im, ax=ax, label="Pearson r", shrink=0.85)
ax.set_title("Δ Metric Correlations — Surgical Co-variation",
             fontsize=12, fontweight="bold")
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, "correlation_heatmap.png"), dpi=120, bbox_inches="tight")
plt.show()
print("[✓] Saved → outputs/correlation_heatmap.png")

In [ ]:
# ── Block 7-E: Random Sample Gallery ────────────────────────────────────────────

GALLERY_N  = 6
sample_ids = random.sample(pair_ids, min(GALLERY_N, len(pair_ids)))

fig, axes = plt.subplots(GALLERY_N, 3, figsize=(18, GALLERY_N * 4.5))
for row, pid in enumerate(sample_ids):
    pb  = cv2.imread(os.path.join(DATA_DIR, f"{pid}_a.jpg"))
    qb  = cv2.imread(os.path.join(DATA_DIR, f"{pid}_b.jpg"))
    if pb is None or qb is None:
        continue
    pr  = cv2.resize(cv2.cvtColor(pb, cv2.COLOR_BGR2RGB), (512, 512))
    qr  = cv2.resize(cv2.cvtColor(qb, cv2.COLOR_BGR2RGB), (512, 512))

    axes[row,0].imshow(analyzer.draw_mesh(pr))
    axes[row,0].set_title(f"Pre-Op  [{pid}]", fontsize=10, color="#2c7bb6", fontweight="bold")
    axes[row,0].axis("off")

    axes[row,1].imshow(analyzer.draw_mesh(qr))
    axes[row,1].set_title(f"Post-Op [{pid}]", fontsize=10, color="#d7191c", fontweight="bold")
    axes[row,1].axis("off")

    if pid in df_delta.index:
        rd = df_delta.loc[pid, KEY]
        clr = ["#d7191c" if v>0 else "#2c7bb6" for v in rd]
        axes[row,2].barh(range(len(rd)), rd.values, color=clr, alpha=.8)
        axes[row,2].set_yticks(range(len(rd)))
        axes[row,2].set_yticklabels([m[:22] for m in rd.index], fontsize=7)
        axes[row,2].axvline(0, color="black", lw=.8, ls="--")
        axes[row,2].set_title(f"Metric Δ [{pid}]", fontsize=10, fontweight="bold")

plt.suptitle("Dataset Gallery — Pre/Post Pairs with Anthropometric Deltas",
             fontsize=13, fontweight="bold", y=1.005)
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, "gallery.png"), dpi=100, bbox_inches="tight")
plt.show()
print("[✓] Saved → outputs/gallery.png")

In [ ]:
# ── Block 7-F: Surgeon Replication Fidelity Score ───────────────────────────────
# Measures metric agreement between GAN-generated post-op and real post-op.
# Only runs a small subset (GAN synthesis is slow without GPU).

FIDELITY_N = 5 if DEVICE == "cpu" else 20
test_ids   = random.sample(pair_ids, min(FIDELITY_N, len(pair_ids)))
fid_rows   = []

for pid in tqdm(test_ids, desc="Fidelity"):
    pp = os.path.join(DATA_DIR, f"{pid}_a.jpg")
    qp = os.path.join(DATA_DIR, f"{pid}_b.jpg")
    pb, qb = cv2.imread(pp), cv2.imread(qp)
    if pb is None or qb is None: continue

    pre  = cv2.resize(cv2.cvtColor(pb, cv2.COLOR_BGR2RGB), (1024, 1024))
    real = cv2.resize(cv2.cvtColor(qb, cv2.COLOR_BGR2RGB), (512,  512))

    # Build slider proxy from actual dataset delta
    sv = {}
    if pid in df_delta.index:
        r = df_delta.loc[pid]
        sv = {
            "Nose Bridge (Hump)"    : float(np.clip(r.get("Nose Length", 0)*5,    -3,3)),
            "Alar Width (Flare)"    : float(np.clip(r.get("Alar Base Width",0)*5, -3,3)),
            "Jawline Width (V-line)": float(np.clip(r.get("Jaw Width",0)*5,       -3,3)),
            "Lip Fullness"          : float(np.clip(r.get("Lip Height",0)*8,      -3,3)),
        }

    w_base  = encoder.encode(pre)
    w_new   = apply_surgical_edits(w_base, sv)
    generator._stub_img = pre
    gen_img = cv2.resize(generator.synthesize(w_new), (512, 512))

    lm_r = analyzer.get_landmarks(real)
    lm_g = analyzer.get_landmarks(gen_img)
    if lm_r is None or lm_g is None: continue

    mr, mg = analyzer.compute_metrics(lm_r), analyzer.compute_metrics(lm_g)
    common = set(mr) & set(mg)
    mae    = np.mean([abs(mr[k]-mg[k])/(abs(mr[k])+1e-8) for k in common])
    fid_rows.append({"Pair ID":pid, "Fidelity":round(1-mae, 4), "N":len(common)})

if fid_rows:
    df_fid = pd.DataFrame(fid_rows).set_index("Pair ID")
    print(f"\n── Surgeon Replication Fidelity (n={len(fid_rows)}) ──")
    display(df_fid.sort_values("Fidelity", ascending=False))
    print(f"\n  Mean: {df_fid['Fidelity'].mean():.3f}  |  1.0 = perfect")
    df_fid.to_csv(os.path.join(OUTPUT_DIR, "fidelity.csv"))
else:
    print("[!] No fidelity data — check face detection")

## Block 8 — Quick Demo: Simulate + Compare vs. Real Ground Truth

In [ ]:
# ── Block 8: Quick Demo ──────────────────────────────────────────────────────────
# Change DEMO_ID or DEMO_SLIDERS and re-run this cell.

DEMO_SLIDERS = {
    "Nose Bridge (Hump)"    : -1.5,   # reduce dorsal hump
    "Nose Tip Rotation"     :  1.0,   # lift tip
    "Alar Width (Flare)"    : -0.5,   # narrow base
    "Jawline Width (V-line)": -1.0,   # V-line effect
    "Chin Projection"       :  0.5,
    "Lip Fullness"          :  0.8,
    "Cheek Volume"          :  0.0,
    "Eye Opening (Bleph.)"  :  0.0,
    "Face Length"           :  0.0,
    "Forehead Slope"        :  0.0,
}

pp = os.path.join(DATA_DIR, f"{DEMO_ID}_a.jpg")
qp = os.path.join(DATA_DIR, f"{DEMO_ID}_b.jpg")

if not os.path.exists(pp):
    print(f"[!] {pp} not found — change DEMO_ID in Block 0")
else:
    print(f"Running demo for pair {DEMO_ID}...")
    pre, sim_post = full_pipeline(pp, DEMO_SLIDERS)

    # Real post-op ground truth
    qb = cv2.imread(qp)
    real_post = cv2.resize(cv2.cvtColor(qb, cv2.COLOR_BGR2RGB), (1024,1024)) \
                if qb is not None else None

    # Metrics
    _, _, df = analyzer.analyze_pair(pre, sim_post)

    # Figure
    n  = 3 if real_post is not None else 2
    fig, ax = plt.subplots(1, n, figsize=(8*n, 8))
    ax[0].imshow(cv2.resize(analyzer.draw_mesh(pre),      (512,512))); ax[0].axis("off")
    ax[0].set_title(f"Pre-Op [{DEMO_ID}]",  fontsize=12, fontweight="bold", color="#2c7bb6")
    ax[1].imshow(cv2.resize(analyzer.draw_mesh(sim_post), (512,512))); ax[1].axis("off")
    ax[1].set_title("Simulated Post-Op",    fontsize=12, fontweight="bold", color="#d7191c")
    if real_post is not None:
        ax[2].imshow(cv2.resize(analyzer.draw_mesh(real_post),(512,512))); ax[2].axis("off")
        ax[2].set_title("Real Post-Op (GT)",fontsize=12, fontweight="bold", color="#1a9641")
    plt.suptitle(f"Surgical Simulation Demo — Pair {DEMO_ID}",
                 fontsize=14, fontweight="bold")
    plt.tight_layout()
    sp = os.path.join(OUTPUT_DIR, f"demo_{DEMO_ID}.png")
    plt.savefig(sp, dpi=120, bbox_inches="tight")
    plt.show()

    if df is not None:
        print(f"\n── Anthropometric Delta — Pair {DEMO_ID} ──")
        display(df.style
            .applymap(lambda v: "color:#d7191c;font-weight:bold" if float(v)>0
                               else "color:#2c7bb6;font-weight:bold",
                      subset=["Δ (%)"])
            .format({"Pre-Op":"{:.4f}","Post-Op":"{:.4f}",
                     "Δ (abs)":"{:+.4f}","Δ (%)":"{:+.2f}%"})
        )
    print(f"[✓] Saved → {sp}")

---
## Appendix — Architecture & Kaggle Notes

### Model Stack
| Component | Model | Output |
|---|---|---|
| Face Detector | YOLOv8n-face → dlib HOG | Bounding box |
| Face Aligner | dlib 68-pt + affine warp | 1024×1024 FFHQ-aligned RGB |
| Encoder | e4e (pSp backbone, FFHQ) | 18 × 512 W+ latent |
| Surgery | GANSpace PCA @ W+ layers | Edited W+ latent |
| Generator | StyleGAN2-Ada FFHQ | 1024×1024 RGB |
| Measurement | MediaPipe FaceMesh 468 | 16 clinical metrics |

### Kaggle Tips
- **First run:** dlib pip-install compiles from source — takes ~2 min, normal.
- **StyleGAN2 first synthesis:** compiles CUDA kernels via ninja — takes ~1 min, then cached.
- **Session persistence:** Weights in `/kaggle/working/models/` survive the session but not across sessions. Use `Save Version` to keep them.
- **GPU memory:** e4e (1 GB) + SG2 generator (1.2 GB) + activations ≈ 4 GB total. T4 16 GB is sufficient.
- **Outputs:** All figures + CSVs saved to `/kaggle/working/outputs/` — visible in the Output tab after `Save Version`.

### Slider → GANSpace Mapping
| Slider | PC | Layers | Anatomy |
|---|---|---|---|
| Nose Bridge | 10 | 4–8 | Dorsal hump |
| Nose Tip | 11 | 5–9 | Tip rotation/projection |
| Alar Width | 12 | 4–8 | Base narrowing |
| Jawline | 14 | 8–14 | V-line / gonial angle |
| Chin | 15 | 9–14 | Projection / height |
| Cheek Volume | 16 | 6–12 | Augmentation |
| Eye Opening | 7 | 3–7 | Blepharoplasty |
| Lip Fullness | 17 | 9–13 | Augmentation |
| Face Length | 5 | 2–6 | Vertical proportion |
| Forehead | 3 | 2–5 | Height / slope |

In [ ]:
# ── Block 9: Gradio Demo ─────────────────────────────────────────────────────────
# Tab 1 — Surgical Preview: landmark-warp (Gaussian displacement field via
#   cv2.remap). No GAN, no encoder, works on CPU, results in ~200 ms.
#   MediaPipe detects 468 landmarks; sliders drive per-region displacements;
#   a smooth Gaussian weight field blends edits into the photo naturally.
# Tab 2 — Landmark Analysis: mesh overlay + 16 metrics
# Tab 3 — Before / After: 3-panel comparison + delta table

import gradio as gr
import numpy as np, cv2, pandas as pd, matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from PIL import Image as PILImage

print(f"[ok] Gradio {gr.__version__}")

# ─────────────────────────────────────────────────────────────────────────────────
# Warp engine
# ─────────────────────────────────────────────────────────────────────────────────

# Each entry: list of (landmark_name, (dx_direction, dy_direction))
# Positive slider = direction as written; negative slider = opposite.
# scale controls displacement magnitude as fraction of image width.
# sigma controls the radius of influence as fraction of image diagonal.
WARP_MAP = {
    "Nose Bridge"       : {"pts": [("nose_bridge_top",(0,-1)),
                                    ("nose_bridge_mid",(0,-0.6))],
                           "scale": 0.035, "sigma": 0.07},
    "Nose Tip Rotation" : {"pts": [("nose_tip",(0,1)),
                                    ("philtrum",(0,0.4))],
                           "scale": 0.030, "sigma": 0.06},
    "Alar Width"        : {"pts": [("alar_left",(-1,0)),
                                    ("alar_right",(1,0))],
                           "scale": 0.030, "sigma": 0.06},
    "Jawline (V-line)"  : {"pts": [("jaw_gonion_L",(1,0)),
                                    ("jaw_gonion_R",(-1,0)),
                                    ("jaw_left",(0.5,0)),
                                    ("jaw_right",(-0.5,0))],
                           "scale": 0.035, "sigma": 0.09},
    "Chin Projection"   : {"pts": [("chin_tip",(0,1))],
                           "scale": 0.030, "sigma": 0.07},
    "Cheek Volume"      : {"pts": [("lcheek",(-0.7,-0.3)),
                                    ("rcheek",(0.7,-0.3))],
                           "scale": 0.035, "sigma": 0.09},
    "Eye Opening"       : {"pts": [("leye_top",(0,-1)),("leye_bot",(0,1)),
                                    ("reye_top",(0,-1)),("reye_bot",(0,1))],
                           "scale": 0.018, "sigma": 0.05},
    "Lip Fullness"      : {"pts": [("lip_top",(0,-1)),("lip_bot",(0,1))],
                           "scale": 0.020, "sigma": 0.05},
    "Face Length"       : {"pts": [("chin_tip",(0,1)),("forehead",(0,-0.5))],
                           "scale": 0.040, "sigma": 0.10},
    "Forehead Slope"    : {"pts": [("forehead",(0,-1))],
                           "scale": 0.030, "sigma": 0.08},
}

def _gaussian_warp(img, src_list, dst_list, sigma):
    """Apply smooth warp: each control point displaces nearby pixels via
    a Gaussian weight, accumulated into a cv2.remap displacement field."""
    h, w = img.shape[:2]
    yy, xx = np.mgrid[0:h, 0:w].astype(np.float32)
    fdx = np.zeros((h, w), dtype=np.float32)
    fdy = np.zeros((h, w), dtype=np.float32)
    for (sx, sy), (tx, ty) in zip(src_list, dst_list):
        ddx, ddy = tx - sx, ty - sy
        if abs(ddx) < 0.1 and abs(ddy) < 0.1:
            continue
        g = np.exp(-((xx - sx)**2 + (yy - sy)**2) / (2 * sigma**2))
        fdx += ddx * g
        fdy += ddy * g
    map_x = np.clip(xx - fdx, 0, w - 1)
    map_y = np.clip(yy - fdy, 0, h - 1)
    return cv2.remap(img, map_x, map_y, cv2.INTER_LINEAR,
                     borderMode=cv2.BORDER_REPLICATE)

def apply_warp_sliders(img_rgb, landmarks, slider_dict):
    """Given detected landmarks and slider values, warp the image."""
    if landmarks is None:
        return img_rgb
    h, w = img_rgb.shape[:2]
    diag = np.sqrt(h**2 + w**2)
    result = img_rgb.copy()
    for name, val in slider_dict.items():
        if abs(val) < 0.01 or name not in WARP_MAP:
            continue
        cfg  = WARP_MAP[name]
        sig  = cfg["sigma"] * diag
        scl  = cfg["scale"] * w * val
        src, dst = [], []
        for lm_name, (dx, dy) in cfg["pts"]:
            if lm_name not in landmarks:
                continue
            sx, sy = float(landmarks[lm_name][0]), float(landmarks[lm_name][1])
            src.append((sx, sy))
            dst.append((sx + dx * scl, sy + dy * scl))
        if src:
            result = _gaussian_warp(result, src, dst, sig)
    return result

# ─────────────────────────────────────────────────────────────────────────────────
# Gradio callback functions
# ─────────────────────────────────────────────────────────────────────────────────

SZ = 512   # display/processing size

def fn_detect(image):
    """Detect landmarks on uploaded photo; store image+landmarks in state."""
    if image is None:
        return None, None, "Upload a photo first."
    rgb = cv2.resize(np.array(image.convert("RGB")), (SZ, SZ))
    lm  = analyzer.get_landmarks(rgb)
    if lm is None:
        return PILImage.fromarray(rgb), None, \
            "No face detected — try a clearer frontal photo."
    mesh = analyzer.draw_mesh(rgb)
    n = len(lm)
    return (PILImage.fromarray(mesh),
            {"img": rgb, "lm": lm},
            f"Detected {n} landmark groups. Adjust sliders then click Preview.")

def fn_preview(state,
               s_nose_bridge, s_nose_tip, s_alar,
               s_jaw, s_chin, s_cheek,
               s_eye, s_lip, s_face_len, s_forehead):
    """Apply warp sliders to the stored image and return the result."""
    blank = PILImage.fromarray(np.full((SZ, SZ, 3), 30, dtype=np.uint8))
    if state is None:
        return blank
    sliders = {
        "Nose Bridge"      : s_nose_bridge,
        "Nose Tip Rotation": s_nose_tip,
        "Alar Width"       : s_alar,
        "Jawline (V-line)" : s_jaw,
        "Chin Projection"  : s_chin,
        "Cheek Volume"     : s_cheek,
        "Eye Opening"      : s_eye,
        "Lip Fullness"     : s_lip,
        "Face Length"      : s_face_len,
        "Forehead Slope"   : s_forehead,
    }
    warped = apply_warp_sliders(state["img"], state["lm"], sliders)
    # side-by-side: original | warped
    panel = np.concatenate([state["img"], warped], axis=1)
    return PILImage.fromarray(panel)

def fn_single(image):
    if image is None:
        return None, pd.DataFrame({"Note": ["Upload a photo first."]})
    rgb  = cv2.resize(np.array(image.convert("RGB")), (SZ, SZ))
    lm   = analyzer.get_landmarks(rgb)
    mesh = analyzer.draw_mesh(rgb)
    mets = analyzer.compute_metrics(lm)
    if not mets:
        return PILImage.fromarray(mesh), pd.DataFrame(
            {"Note": ["No face detected. Try a clearer frontal photo."]})
    df = pd.DataFrame(
        [{"Metric": k, "Value (IOD-norm)": round(v, 4)} for k, v in mets.items()])
    return PILImage.fromarray(mesh), df

def fn_pair(pre_image, post_image):
    if pre_image is None or post_image is None:
        return None, pd.DataFrame({"Note": ["Upload both images first."]})
    pre = cv2.resize(np.array(pre_image.convert("RGB")), (SZ, SZ))
    pst = cv2.resize(np.array(post_image.convert("RGB")), (SZ, SZ))
    lm_pre, lm_post = analyzer.get_landmarks(pre), analyzer.get_landmarks(pst)
    pre_mesh, pst_mesh = analyzer.draw_mesh(pre), analyzer.draw_mesh(pst)
    pm, qm = analyzer.compute_metrics(lm_pre), analyzer.compute_metrics(lm_post)
    if not pm or not qm:
        fig, (a1, a2) = plt.subplots(1, 2, figsize=(12, 5))
        a1.imshow(pre_mesh); a1.axis("off")
        a1.set_title("Pre-Op",  fontweight="bold", color="#2c7bb6", fontsize=12)
        a2.imshow(pst_mesh); a2.axis("off")
        a2.set_title("Post-Op", fontweight="bold", color="#d7191c", fontsize=12)
        plt.tight_layout()
        return fig, pd.DataFrame({"Note": ["Face not detected in one or both images."]})
    rows = []
    for k in pm:
        pv, qv = pm[k], qm.get(k, float("nan"))
        dv = qv - pv
        rows.append({"Metric": k, "Pre-Op": round(pv,4), "Post-Op": round(qv,4),
                     "Delta": round(dv,4),
                     "Delta %": round(dv/abs(pv)*100 if abs(pv)>1e-6 else 0, 2)})
    df = pd.DataFrame(rows)
    fig = plt.figure(figsize=(20, 7))
    gs  = fig.add_gridspec(1, 3, width_ratios=[2, 2, 3])
    ax1, ax2, ax3 = (fig.add_subplot(gs[i]) for i in range(3))
    ax1.imshow(pre_mesh); ax1.axis("off")
    ax1.set_title("Pre-Op + Landmarks",  fontsize=12, fontweight="bold", color="#2c7bb6")
    ax2.imshow(pst_mesh); ax2.axis("off")
    ax2.set_title("Post-Op + Landmarks", fontsize=12, fontweight="bold", color="#d7191c")
    d   = df["Delta"].astype(float)
    clr = ["#d7191c" if v > 0 else "#2c7bb6" for v in d]
    ax3.barh(range(len(d)), d.values, color=clr, alpha=0.85, ec="white")
    ax3.set_yticks(range(len(d))); ax3.set_yticklabels(df["Metric"], fontsize=9)
    ax3.axvline(0, color="black", lw=0.8, ls="--")
    ax3.set_xlabel("Delta (IOD-normalised)", fontsize=10)
    ax3.set_title("Anthropometric Changes", fontsize=11, fontweight="bold")
    for i, (v, pct) in enumerate(zip(d, df["Delta %"])):
        ax3.text(float(v)+(0.002 if float(v)>=0 else -0.002), i,
                 f"{pct:+.1f}%", va="center", fontsize=7.5,
                 ha="left" if float(v)>=0 else "right")
    ax3.legend(handles=[mpatches.Patch(color="#d7191c", label="Increase"),
                        mpatches.Patch(color="#2c7bb6", label="Decrease")],
               fontsize=9, loc="lower right")
    plt.tight_layout()
    return fig, df

# ─────────────────────────────────────────────────────────────────────────────────
# Build UI
# ─────────────────────────────────────────────────────────────────────────────────

with gr.Blocks(title="Surgical Outcome Simulator", theme=gr.themes.Soft()) as demo:

    gr.Markdown(
        "## Facial Anthropometric Surgical Simulator\n"
        "Landmark detection | Anthropometric measurement | Surgical preview"
    )

    # ── Tab 1: Surgical Preview (warp-based, no GAN) ──────────────────────────────
    with gr.Tab("Surgical Preview"):
        gr.Markdown(
            "**Step 1** — Upload a frontal photo and click **Detect** (~0.2 s).\n\n"
            "**Step 2** — Adjust sliders then click **Preview** (~0.2 s).\n\n"
            "Works on CPU — no GPU required. Results appear on the right "
            "(original left | simulated right)."
        )
        face_state = gr.State(value=None)
        with gr.Row():
            t3_inp    = gr.Image(label="Input Photo", type="pil")
            t3_mesh   = gr.Image(label="Detected Landmarks")
        t3_status    = gr.Textbox(label="Status", interactive=False, max_lines=2)
        t3_detect    = gr.Button("Step 1: Detect Landmarks", variant="secondary")

        gr.Markdown("---")
        gr.Markdown("### Surgical Sliders  (negative = reduce / reverse)")
        with gr.Row():
            with gr.Column():
                sl_nose_bridge = gr.Slider(-3, 3, 0, step=0.1, label="Nose Bridge (hump)")
                sl_nose_tip    = gr.Slider(-3, 3, 0, step=0.1, label="Nose Tip Rotation")
                sl_alar        = gr.Slider(-3, 3, 0, step=0.1, label="Alar Width")
                sl_jaw         = gr.Slider(-3, 3, 0, step=0.1, label="Jawline (V-line)")
                sl_chin        = gr.Slider(-3, 3, 0, step=0.1, label="Chin Projection")
            with gr.Column():
                sl_cheek    = gr.Slider(-3, 3, 0, step=0.1, label="Cheek Volume")
                sl_eye      = gr.Slider(-3, 3, 0, step=0.1, label="Eye Opening")
                sl_lip      = gr.Slider(-3, 3, 0, step=0.1, label="Lip Fullness")
                sl_face_len = gr.Slider(-3, 3, 0, step=0.1, label="Face Length")
                sl_forehead = gr.Slider(-3, 3, 0, step=0.1, label="Forehead Slope")

        t3_preview = gr.Button("Step 2: Preview", variant="primary")
        t3_result  = gr.Image(label="Original  |  Simulated")

        t3_detect.click(fn=fn_detect, inputs=[t3_inp],
                        outputs=[t3_mesh, face_state, t3_status])
        t3_preview.click(fn=fn_preview,
                         inputs=[face_state,
                                 sl_nose_bridge, sl_nose_tip, sl_alar,
                                 sl_jaw, sl_chin, sl_cheek,
                                 sl_eye, sl_lip, sl_face_len, sl_forehead],
                         outputs=[t3_result])

    # ── Tab 2: Landmark Analysis ──────────────────────────────────────────────────
    with gr.Tab("Landmark Analysis"):
        gr.Markdown("Upload a frontal face photo for landmark overlay and 16 clinical metrics.")
        with gr.Row():
            t1_inp  = gr.Image(label="Input Photo", type="pil")
            t1_mesh = gr.Image(label="Landmark Overlay")
        t1_table = gr.Dataframe(label="Clinical Metrics (IOD-normalised)")
        gr.Button("Analyze", variant="primary").click(
            fn=fn_single, inputs=t1_inp, outputs=[t1_mesh, t1_table])

    # ── Tab 3: Before / After ─────────────────────────────────────────────────────
    with gr.Tab("Before / After"):
        gr.Markdown("Upload a pre-op and post-op photo for the 3-panel comparison.")
        with gr.Row():
            t2_pre  = gr.Image(label="Pre-Op  (Before)", type="pil")
            t2_post = gr.Image(label="Post-Op (After)",  type="pil")
        t2_plot  = gr.Plot(label="Comparison Figure")
        t2_delta = gr.Dataframe(label="Metric Deltas")
        gr.Button("Compare", variant="primary").click(
            fn=fn_pair, inputs=[t2_pre, t2_post], outputs=[t2_plot, t2_delta])

# ── Launch ────────────────────────────────────────────────────────────────────────

demo.queue()
demo.launch(share=True, quiet=False)
print("[ok] Gradio running — copy the gradio.live URL above to share")